In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/repo')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'feat/shared-signal-poc', 'https://github.com/huongiang171/green-sm-foundation-pretraining.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)
print('Repo:', repo)


In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(repo))
from scripts.package_shared_signal_poc import extract_shared_signal_poc
input_root = Path('/kaggle/input')
payloads = list(input_root.rglob('*.tar.gz.b64'))
assert len(payloads) == 1, f'Attach exactly one shared-signal payload; found {payloads}'
corpus_root = Path('/tmp/green_sm_shared_signal_poc_v1')
extract_shared_signal_poc(payloads[0], corpus_root)
linked = corpus_root / 'green_sm_shared_signal_poc_linked_v1' / 'foundation'
control = corpus_root / 'green_sm_shared_signal_poc_control_paired_v1' / 'foundation'
assert (linked / 'manifest.json').is_file() and (control / 'manifest.json').is_file()
config = json.loads((repo / 'configs/pretrain_green_sm_shared_signal_poc.json').read_text())
config['linked_root'], config['control_root'] = str(linked), str(control)
config['output_dir'] = '/kaggle/working/artifacts/green_sm_shared_signal_poc_v1'
runtime = Path('/kaggle/working/pretrain_shared_signal_runtime.json')
runtime.write_text(json.dumps(config, indent=2))
print('Payload:', payloads[0], 'linked:', linked, 'runtime:', runtime, flush=True)


In [ ]:
%cd /kaggle/working/repo
!python -u -m src.pretraining --config /kaggle/working/pretrain_shared_signal_runtime.json --corpus linked --device cuda


In [ ]:
!find /kaggle/working/artifacts/green_sm_shared_signal_poc_v1 -maxdepth 4 -type f | sort
!test -f /kaggle/working/artifacts/green_sm_shared_signal_poc_v1/*/linked/checkpoint_manifest.json
!cd /kaggle/working/artifacts/green_sm_shared_signal_poc_v1 && tar -czf /kaggle/working/green_sm_shared_signal_poc_checkpoint.tar.gz .
!ls -lh /kaggle/working/green_sm_shared_signal_poc_checkpoint.tar.gz
